# DB1B Airline Analysis: Python EDA and Visualization
Charts built from the summary CSVs in `outputs/`, which were exported from PostgreSQL. The raw tables have tens of millions of rows, so they stay in the database.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("../outputs")
IMAGE_DIR = Path("../images")
IMAGE_DIR.mkdir(exist_ok=True)

quarter = pd.read_csv(OUTPUT_DIR / "quarter_summary.csv")
distance = pd.read_csv(OUTPUT_DIR / "distance_group_summary.csv")
carrier = pd.read_csv(OUTPUT_DIR / "carrier_summary.csv")
trip = pd.read_csv(OUTPUT_DIR / "trip_type_summary.csv")
coupons = pd.read_csv(OUTPUT_DIR / "coupon_summary.csv")
state = pd.read_csv(OUTPUT_DIR / "origin_state_summary.csv")
share = pd.read_csv(OUTPUT_DIR / "carrier_share_ca_tx.csv")
segment = pd.read_csv(OUTPUT_DIR / "segment_carrier_summary.csv")


## 1. Check the CSVs
Shape, dtypes and missing values for each file.

In [ ]:
datasets = {
    "quarter": quarter,
    "distance": distance,
    "carrier": carrier,
    "trip": trip,
    "coupons": coupons,
    "state": state,
    "share": share,
    "segment": segment,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")
    print(df.dtypes)
    print("Missing values:", int(df.isna().sum().sum()))
    print()


## 2. Quarter analysis

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(quarter["quarter"], quarter["avg_itin_fare"], marker="o")
plt.title("Average itinerary fare by quarter")
plt.xlabel("Quarter")
plt.ylabel("Average itinerary fare")
plt.xticks([1,2,3,4])
plt.tight_layout()
plt.savefig(IMAGE_DIR / "quarter_avg_fare.png", dpi=180)
plt.show()


Q4 has the highest average fare (about 457) and Q3 the lowest (about 422).

## 3. Distance and fare

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(distance["distance_group"], distance["avg_itin_fare"], marker="o")
plt.title("Average itinerary fare by distance group")
plt.xlabel("Distance group")
plt.ylabel("Average itinerary fare")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "distance_group_avg_fare.png", dpi=180)
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(distance["distance_group"], distance["avg_fpm"], marker="o")
plt.title("Average fare per mile by distance group")
plt.xlabel("Distance group")
plt.ylabel("Average fare per mile")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "distance_group_avg_fpm.png", dpi=180)
plt.show()


Average fare generally rises with distance group while fare per mile falls. The longest-distance groups have far fewer itineraries.

## 4. Carrier volume

In [ ]:
top10 = carrier.nlargest(10, "total_itin").sort_values("total_itin")
plt.figure(figsize=(9,6))
plt.barh(top10["rp_carrier"], top10["total_itin"])
plt.title("Top 10 carriers by itinerary count")
plt.xlabel("Itinerary count")
plt.ylabel("Carrier")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "top_carriers_volume.png", dpi=180)
plt.show()


WN has the most itineraries in 2024.

## 5. One-way vs round-trip

In [ ]:
trip_plot = trip.copy()
trip_plot["trip_type"] = trip_plot["round_trip"].map({0: "One-way", 1: "Round-trip"})
plt.figure(figsize=(7,5))
plt.bar(trip_plot["trip_type"], trip_plot["avg_itin_fare"])
plt.title("Average itinerary fare by trip type")
plt.ylabel("Average itinerary fare")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "trip_type_avg_fare.png", dpi=180)
plt.show()


Round trips are more common, longer and more expensive in total, but slightly cheaper per mile than one-way.

## 6. Coupon distribution

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(coupons["coupons"].astype(str), coupons["total_itin"])
plt.title("Itinerary count by number of coupons")
plt.xlabel("Coupons")
plt.ylabel("Itinerary count")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "coupon_distribution.png", dpi=180)
plt.show()


Two-coupon itineraries are the largest group.

## 7. WN carrier share in CA and TX

In [ ]:
plt.figure(figsize=(8,5))
for state_name in ["CA", "TX"]:
    temp = share[(share["rp_carrier"] == "WN") & (share["origin_state"] == state_name)].sort_values("quarter")
    plt.plot(temp["quarter"], temp["carrier_share"], marker="o", label=state_name)
plt.title("WN carrier share in CA and TX")
plt.xlabel("Quarter")
plt.ylabel("Carrier share (%)")
plt.xticks([1,2,3,4])
plt.legend()
plt.tight_layout()
plt.savefig(IMAGE_DIR / "wn_share_ca_tx.png", dpi=180)
plt.show()


WN is the largest carrier in both states in every quarter.

## 8. Round-trip segment distance

In [ ]:
top_segment = segment.nlargest(10, "avg_segment_distance").sort_values("avg_segment_distance")
plt.figure(figsize=(9,6))
plt.barh(top_segment["rp_carrier"], top_segment["avg_segment_distance"])
plt.title("Top carriers by average round-trip segment distance")
plt.xlabel("Average segment distance")
plt.ylabel("Carrier")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "segment_distance_top_carriers.png", dpi=180)
plt.show()


HA has the longest average segment on round-trip itineraries.

## Limitations
The CSVs only hold counts and averages, so medians and quartiles can't be computed here. Those would have to be calculated in PostgreSQL first.